In [ ]:
import chess
import torch
import torch.nn as nn
import torch.nn.functional as f
import pygame

if torch.cuda.is_available():
	print("PyTorch is using the GPU")
	GPUCount = torch.cuda.device_count()
	print(f"Found {GPUCount} GPUs")

	for i in range(GPUCount):
		print(f"GPU {i} found: {torch.cuda.get_device_name(i)}")

	device = torch.device("cuda:0")
else:
	print("PyTorch is using the CPU")
	device = torch.device("cpu")

print(f"Selected Device: {device}")

In [ ]:
pygame.init()
BoardSize = 512
SquareSize = BoardSize//8
DarkColor = (90,90,90)
LightColor = (205,205,205)
ChessBoard = chess.Board()
ChessBoardGUI = pygame.display.set_mode((BoardSize,BoardSize))
turn = 0
images = {}
ranks = [8,7,6,5,4,3,2,1]
files = ['a','b','c','d','e','f','g','h']
EvalCache = {}
MVVLVALookup = [[0,0,0,0,0,0,0],
				[0,9,29,29,49,89,0],
				[0,7,17,17,47,87,0],
				[0,7,17,17,47,87,0],
				[0,5,15,15,45,85,0],
				[0,1,11,11,41,81,0],
				[0,0,0,0,0,0,0]
				]

In [ ]:
def MoveSort(move):
	if ChessBoard.is_en_passant(move):
		return 9
	elif ChessBoard.is_capture(move):
		return MVVLVALookup[ChessBoard.piece_at(move.from_square).piece_type][ChessBoard.piece_at(move.to_square).piece_type]
	else:
		return 0

In [ ]:
def LoadImages(SquareSize):
	pieces = {'K','Q','B','N','R','P','k','q','b','n','r','p'}
	ImgMap = {'K':'WhiteKing','Q':'WhiteQueen','B':'WhiteBishop','N':'WhiteKnight','R':'WhiteRook','P':'WhitePawn','k':'BlackKing','q':'BlackQueen','b':'BlackBishop','n':'BlackKnight','r':'BlackRook','p':'BlackPawn'}

	for piece in pieces:
		img = pygame.image.load(f'ChessSprites/{ImgMap[piece]}.png')
		img = pygame.transform.scale(img,(SquareSize,SquareSize))
		images[piece] = img
	return

In [ ]:
def DrawBoard(ChessBoardGUI,SquareSize):
	for row in range(8):
		for col in range(8):
			x = col*SquareSize
			y = row*SquareSize
			Color = LightColor if ((row+col)%2==0) else DarkColor
			pygame.draw.rect(ChessBoardGUI,Color,(x,y,SquareSize,SquareSize))
	return

In [ ]:
def DrawPieces(ChessBoardGUI,ChessBoard):
	for row in range(8):
		for col in range(8):
			x = col*SquareSize
			y = row*SquareSize
			SquareIdx = chess.square(col,7-row)
			piece = ChessBoard.piece_at(SquareIdx)
			if (piece is not None):
				ChessBoardGUI.blit(images[piece.symbol()],(x,y))

In [ ]:
def ProcessChessData(FENString):
	tensor = torch.zeros((14,8,8), dtype=torch.float32)
	board = chess.Board(FENString)

	PieceToLayer = {
		'P':0,'N':1,'B':2,'R':3,'Q':4,'K':5,
		'p':6,'n':7,'b':8,'r':9,'q':10,'k':11
	}

	for square in chess.SQUARES:
		piece = board.piece_at(square)

		if piece:
			symbol = piece.symbol()
			layer = PieceToLayer[symbol]

			row = 7-(square//8)
			col = square%8

			tensor[layer,row,col] = 1.0

	if board.turn == chess.WHITE:
		tensor[12,:,:] = 1.0

	if board.has_kingside_castling_rights(chess.WHITE):
		tensor[13,7,7] = 1.0
	if board.has_queenside_castling_rights(chess.WHITE):
		tensor[13,7,0] = 1.0
	if board.has_kingside_castling_rights(chess.BLACK):
		tensor[13,0,7] = 1.0
	if board.has_queenside_castling_rights(chess.BLACK):
		tensor[13,0,0] = 1.0

	return tensor

In [ ]:
class SEBlock(nn.Module):
	def __init__(self,channels,reduction=16):
		super().__init__()
		self.squeeze = nn.AdaptiveAvgPool2d(1)
		self.excite = nn.Sequential(
			nn.Linear(channels,channels//reduction,bias=False),
			nn.SiLU(inplace=True),
			nn.Linear(channels//reduction,channels,bias=False),
			nn.Sigmoid()
		)

	def forward(self,x):
		b,c,_,_ = x.size()
		y = self.squeeze(x).view(b,c)
		y = self.excite(y).view(b,c,1,1)
		return x * y.expand_as(x)

class ResidualBlock(nn.Module):
	def __init__(self,NumChannels):
		super().__init__()
		self.conv1 = nn.Conv2d(NumChannels,NumChannels,kernel_size=3,padding=1)
		self.bn1 = nn.BatchNorm2d(NumChannels)

		self.conv2 = nn.Conv2d(NumChannels,NumChannels,kernel_size=3,padding=1)
		self.bn2 = nn.BatchNorm2d(NumChannels)

		self.se = SEBlock(NumChannels)

	def forward(self,x):
		residual = x
		
		x = f.silu(self.bn1(self.conv1(x)))

		x = self.bn2(self.conv2(x))

		x = self.se(x)

		x += residual

		return f.silu(x)
	
class ChessNet(nn.Module):
	def __init__(self):
		super().__init__()

		self.ConvInput = nn.Conv2d(in_channels=14,out_channels=256,kernel_size=3,padding=1)
		self.BnInput = nn.BatchNorm2d(256)

		self.ResTower = nn.Sequential(*[ResidualBlock(256) for _ in range(10)])

		self.ConvValue = nn.Conv2d(in_channels=256,out_channels=32,kernel_size=1)
		self.BnValue = nn.BatchNorm2d(32)

		self.flat = nn.Flatten()

		self.fc1 = nn.Linear(32*8*8,256)
		self.fc2 = nn.Linear(256,1)

	def forward(self,x):
		x = f.silu(self.BnInput(self.ConvInput(x)))

		x = self.ResTower(x)

		x = f.silu(self.BnValue(self.ConvValue(x)))
		x = self.flat(x)
		x = f.silu(self.fc1(x))

		x = torch.tanh(self.fc2(x))

		return x
	
model = ChessNet()
model.to(device)
model.load_state_dict(torch.load('ChessModel.pth'))
model.eval()

In [ ]:
def EvalNetwork(board):
	fen = board.fen()
	if fen in EvalCache:
		return EvalCache[fen]
	
	if board.is_checkmate():
		return -9999.0 if board.turn == chess.WHITE else 9999.0
	if board.is_game_over():
		return 0.0
	
	tensor = ProcessChessData(fen).unsqueeze(0).to(device)

	with torch.no_grad():
		score = model(tensor).item()

	EvalCache[fen] = score

	return score

In [ ]:
def minimax(board,depth,alpha,beta,MaximisingPlayer):
	if depth == 0 or board.is_game_over():
		return EvalNetwork(board)
	
	if MaximisingPlayer:
		MaxEval = -float('inf')
		for move in board.legal_moves:
			board.push(move)
			EvalScore = minimax(board,depth-1,alpha,beta,False)
			board.pop()

			MaxEval = max(MaxEval,EvalScore)
			alpha = max(alpha,EvalScore)

			if beta <= alpha:
				break

		return MaxEval
	
	else:
		MinEval = float('inf')
		for move in board.legal_moves:
			board.push(move)
			EvalScore = minimax(board,depth-1,alpha,beta,True)
			board.pop()

			MinEval = min(MinEval,EvalScore)
			beta = min(beta,EvalScore)

			if beta <= alpha:
				break

		return MinEval

In [ ]:
def GetBestMove(board,depth):
	BestMove = None
	MaximisingPlayer = (board.turn == chess.WHITE)

	BestEval = -float('inf') if MaximisingPlayer else float('inf')
	alpha = -float('inf')
	beta = float('inf')

	MoveList = sorted(board.legal_moves,key=MoveSort,reverse=True)

	for move in MoveList:
		board.push(move)
		MoveEval = minimax(board,depth-1,alpha,beta,not MaximisingPlayer)
		board.pop()

		if MaximisingPlayer:
			if MoveEval > BestEval:
				BestEval = MoveEval
				BestMove = move
			alpha = max(alpha,MoveEval)
		else:
			if MoveEval < BestEval:
				BestEval = MoveEval
				BestMove = move
			beta = min(beta,MoveEval)

	return BestMove

In [ ]:
LoadImages(SquareSize)
PlayerClicks = []

try:
	with torch.no_grad():
		while not ChessBoard.is_game_over():
			for event in pygame.event.get():
				if event.type == pygame.QUIT:
					break
				elif event.type == pygame.MOUSEBUTTONDOWN and turn == 0:
					x,y = pygame.mouse.get_pos()
					col = x//SquareSize
					row = y//SquareSize
					ClickedSquare = files[col]+str(ranks[row])
					PlayerClicks.append(ClickedSquare)
					if len(PlayerClicks) == 2:
						if PlayerClicks[0] == PlayerClicks[1]:
							PlayerClicks = []
						else:
							move = PlayerClicks[0]+PlayerClicks[1]
							altmove = move+'q'
							move = chess.Move.from_uci(move)
							altmove = chess.Move.from_uci(altmove)
							if move in ChessBoard.legal_moves:
								ChessBoard.push(move)
								turn = 1
							elif altmove in ChessBoard.legal_moves:
								ChessBoard.push(altmove)
								turn = 1
							PlayerClicks = []
			
			DrawBoard(ChessBoardGUI,SquareSize)
			DrawPieces(ChessBoardGUI,ChessBoard)
			pygame.display.flip()
			
			if turn == 1:
				BestMove = GetBestMove(ChessBoard,3)
				if BestMove is not None:
					ChessBoard.push(BestMove)

				turn = 0

			DrawBoard(ChessBoardGUI,SquareSize)
			DrawPieces(ChessBoardGUI,ChessBoard)
			pygame.display.flip()
finally:
	pygame.quit()

In [ ]:
pygame.quit()